ai usage
help with creation of sql statement

this dataset, is made to predict the values for the daily trips for the whole city

In [15]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from scipy import stats
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()   # finds the .env in the repo root, also from citibike/notebooks/

engine = create_engine(
    "postgresql+psycopg://",
    connect_args={
        "host": "100.88.194.89",
        "port": 5432,
        "dbname": "citibike",
        "user": os.environ["CITIBIKE_DB_USER"],
        "password": os.environ["CITIBIKE_DB_PASSWORD"].strip(),
        "connect_timeout": 10,
    },
)

with engine.connect() as conn:
    print("Connected as", conn.execute(text("SELECT current_user")).scalar())

OperationalError: (psycopg.OperationalError) connection failed: connection to server at "100.88.194.89", port 5432 failed: FATAL:  too many connections for role "r0957896"
(Background on this error at: https://sqlalche.me/e/21/e3q8)

In [ ]:
def fmt(value, spec: str = ",") -> str:
    """Format a number with a dot as thousands separator and a comma as decimal separator: 212.310.903 / 0,057."""
    return f"{value:{spec}}".translate(str.maketrans(",.", ".,"))


def sql(query: str, **params) -> pd.DataFrame:
    t0 = time.time()
    df = pd.read_sql(text(query), engine, params=params)
    print(f"{fmt(len(df))} rows in {fmt(time.time() - t0, '.1f')} s")
    return df

COLORS = {"member": "#2a78d6", "casual": "#eb6834", "classic_bike": "#1baf7a", "electric_bike": "#4a3aa7"}
LABELS = {"member": "Member", "casual": "Casual", "classic_bike": "Classic bike", "electric_bike": "E-bike"}

plt.rcParams.update({
    "figure.figsize": (11, 4.2), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.edgecolor": "#8a8986", "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "lines.linewidth": 2, "legend.frameon": False,
})
millions = FuncFormatter(lambda v, _: f"{fmt(v / 1e6, '.1f')} M")
thousands = FuncFormatter(lambda v, _: f"{v / 1e3:.0f} k")

# Every DataFrame the notebook shows gets the same number format as fmt(). Integer columns get a thousands
# separator; float columns get as many decimals as their values need (at most pandas' display precision).
# Columns that hold a year are left alone, otherwise 2025 would show as 2.025.
NO_SEPARATOR = {"year"}

def _decimals(col: pd.Series) -> int:
    values = col.dropna().to_numpy()
    for d in range(pd.get_option("display.precision")):
        if np.allclose(values, values.round(d)):
            return d
    return pd.get_option("display.precision")

def _formatted_html(df: pd.DataFrame) -> str:
    formats = {}
    for name, col in df.items():
        if name in NO_SEPARATOR or pd.api.types.is_bool_dtype(col):
            continue
        if pd.api.types.is_integer_dtype(col):
            formats[name] = fmt
        elif pd.api.types.is_float_dtype(col):
            formats[name] = lambda v, spec=f",.{_decimals(col)}f": fmt(v, spec)
    return df.style.format(formats, na_rep="").to_html(table_attributes='class="dataframe"')

pd.set_option("styler.render.max_rows", 60)   # a styled table does not truncate itself like a plain one
get_ipython().display_formatter.formatters["text/html"].for_type(pd.DataFrame, _formatted_html)

In [ ]:
first = sql("SELECT * FROM citibike.trips LIMIT 5")
display(first)

5 rows in 0,2 s


,ride_id,started_at,ended_at,duration_s,start_station_id,start_station_name,start_lat,start_lng,end_station_id,end_station_name,end_lat,end_lng,rideable_type,member_casual,region,schema_version
0,2F0248F2E85771EA,2021-01-19 19:43:36.986000,2021-01-19 19:45:50.414000,133,5406.02,Rivington St & Ridge St,"40,7185","-73,983",5414.06,Allen St & Rivington St,"40,72","-73,990",electric_bike,member,nyc,v2
1,49985469DD6C5EC9,2021-01-29 06:38:32.423000,2021-01-29 06:40:28.603000,116,4789.03,Clark St & Henry St,"40,6976","-73,993",4829.01,Columbia Heights & Cranberry St,"40,70","-73,995",classic_bike,member,nyc,v2
2,E3B2362D59B6182D,2021-01-23 06:29:31.470000,2021-01-23 06:31:18.457000,107,4789.03,Clark St & Henry St,"40,6976","-73,993",4829.01,Columbia Heights & Cranberry St,"40,70","-73,995",classic_bike,member,nyc,v2
3,1C82E20D9DB94A58,2021-01-24 06:28:00.463000,2021-01-24 06:29:52.060000,112,4789.03,Clark St & Henry St,"40,6976","-73,993",4829.01,Columbia Heights & Cranberry St,"40,70","-73,995",classic_bike,member,nyc,v2
4,B82510C13F251703,2021-01-09 06:33:39.383000,2021-01-09 06:35:31.520000,112,4789.03,Clark St & Henry St,"40,6976","-73,993",4829.01,Columbia Heights & Cranberry St,"40,70","-73,995",classic_bike,member,nyc,v2


# base frame
The dataframe below has been prepared for a prediction model made to predict the amount of minutes a trip will take

this model consists of:

- start_season : data extracted from the started_at column, this column showcases what season (winter, spring, summer, fall) the bike ride starts
- start_month  : data extracted from the started_at column, this column showcases what month (1-12) the bike ride starts
- start_day_of_week : data extracted from the started_at column, this column showcases what day of the week (0-6) the bike ride starts
- start_time_of_day	: data extracted from the started_at column, this column showcases what time of day (morning, afternoon, evening, night) the bike ride starts
- start_station_name : what station the bike ride starts
- rideable_type	 : what type of bike (electric/classic) the rider is using
- member_casual	: what membership the biker has (casual/member)
- trip_duration_min : the value we're predicting, extracted from the value duration_s & rounded for use, this showcases how many minutes the trip takes
                    - to remove outliers, we have truncated the results to 120 minutes max (7600seconds)


In [ ]:

trips = sql("""
    SELECT

        -- Season
        CASE
            WHEN EXTRACT(MONTH FROM started_at) IN (12, 1, 2)
                THEN 'Winter'
            WHEN EXTRACT(MONTH FROM started_at) IN (3, 4, 5)
                THEN 'Spring'
            WHEN EXTRACT(MONTH FROM started_at) IN (6, 7, 8)
                THEN 'Summer'
            ELSE 'Autumn'
        END AS start_season,

        -- Month
        EXTRACT(MONTH FROM started_at) AS start_month,

        -- Day of the week
        EXTRACT(DOW FROM started_at) AS start_day_of_week,

        -- Time of day
        CASE
            WHEN EXTRACT(HOUR FROM started_at) >= 5
                 AND EXTRACT(HOUR FROM started_at) < 12
                THEN 'Morning'
            WHEN EXTRACT(HOUR FROM started_at) >= 12
                 AND EXTRACT(HOUR FROM started_at) < 17
                THEN 'Afternoon'
            WHEN EXTRACT(HOUR FROM started_at) >= 17
                 AND EXTRACT(HOUR FROM started_at) < 21
                THEN 'Evening'
            ELSE 'Night'
        END AS start_time_of_day,

        start_station_name,
        rideable_type,
        member_casual,

        ROUND(duration_s / 60.0) AS trip_duration_min

    FROM citibike.trips
    WHERE
        duration_s > 0
        AND duration_s <= 7200
        AND started_at IS NOT NULL
        AND start_station_id IS NOT NULL
        AND rideable_type IS NOT NULL
        AND member_casual IS NOT NULL
    LIMIT 50000
""")

trips.head()



50.000 rows in 1,3 s


,start_season,start_month,start_day_of_week,start_time_of_day,start_station_name,rideable_type,member_casual,trip_duration_min
0,Winter,1,1,Afternoon,University Pl & E 8 St,classic_bike,member,6
1,Winter,1,5,Evening,University Pl & E 8 St,classic_bike,casual,5
2,Winter,1,3,Night,Pershing Square North,electric_bike,member,8
3,Winter,1,3,Morning,University Pl & E 8 St,electric_bike,casual,8
4,Winter,1,5,Evening,University Pl & E 8 St,classic_bike,member,11


# Frame with endstation
as you will see in our pycarrot, using just the data above makes it hard to estimate the time accurately, so as a test model to compare, we will also be using this frame

this frame consists of all values seen above, but allong with that an endstation has been added
This station is most likely unknown to us when the rider starts his trip, but it is interesting to see the difference in results later on

In [ ]:

tripsEndStation = sql("""
    SELECT

        -- Season
        CASE
            WHEN EXTRACT(MONTH FROM started_at) IN (12, 1, 2)
                THEN 'Winter'
            WHEN EXTRACT(MONTH FROM started_at) IN (3, 4, 5)
                THEN 'Spring'
            WHEN EXTRACT(MONTH FROM started_at) IN (6, 7, 8)
                THEN 'Summer'
            ELSE 'Autumn'
        END AS start_season,

        -- Month
        EXTRACT(MONTH FROM started_at) AS start_month,

        -- Day of the week
        EXTRACT(DOW FROM started_at) AS start_day_of_week,

        -- Time of day
        CASE
            WHEN EXTRACT(HOUR FROM started_at) >= 5
                 AND EXTRACT(HOUR FROM started_at) < 12
                THEN 'Morning'
            WHEN EXTRACT(HOUR FROM started_at) >= 12
                 AND EXTRACT(HOUR FROM started_at) < 17
                THEN 'Afternoon'
            WHEN EXTRACT(HOUR FROM started_at) >= 17
                 AND EXTRACT(HOUR FROM started_at) < 21
                THEN 'Evening'
            ELSE 'Night'
        END AS start_time_of_day,

        start_station_name,
        end_station_name,
        rideable_type,
        member_casual,

        ROUND(duration_s / 60.0) AS trip_duration_min

    FROM citibike.trips
    WHERE
        duration_s > 0
        AND duration_s <= 7200
        AND started_at IS NOT NULL
        AND start_station_id IS NOT NULL
        AND rideable_type IS NOT NULL
        AND member_casual IS NOT NULL
    LIMIT 50000
""")

tripsEndStation.head()